# Chinook Sales Assistant — notebook-only

This is the Module 5 sales assistant without LangGraph Server, Studio, a browser, or an HTML UI. Run it from top to bottom in Jupyter. It executes the graph in-process and uses only notebook output: text, Markdown, and inline PNG charts.

The original course files remain unchanged in `sales_assistant/` for comparison. This version reuses their database, skills, operating manuals, and local mock MCP mail service. The one intentional output change is the newsletter: it is saved as Markdown (`.md`) and displayed in the notebook instead of producing an HTML page.

**Prerequisites:** run the notebook with the `python` project environment and set the model key(s) selected in `python/models.py`. Set `TAVILY_API_KEY` only if you want the optional direct-API newsletter research. No browser is used.

In [ ]:
# Run from python/m5 so the course files are found predictably.
from pathlib import Path
import os
import sys

M5_DIR = Path.cwd().resolve()
if not (M5_DIR / 'sales_assistant').is_dir():
    raise RuntimeError('Open this notebook with python/m5 as the working directory.')
PYTHON_DIR = M5_DIR.parent
SALES_DIR = M5_DIR / 'sales_assistant'
OUTPUTS_DIR = SALES_DIR / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

os.chdir(SALES_DIR)
sys.path[:0] = [str(M5_DIR), str(PYTHON_DIR), str(SALES_DIR)]
print(f'Working directory: {Path.cwd()}')

In [ ]:
# Load the same course configuration. This prints no secrets.
from dotenv import load_dotenv

# Local notebook credentials/configuration take precedence over the course-wide file.
load_dotenv(M5_DIR / '.env', override=True)
print('OPENAI_API_KEY configured:', bool(os.environ.get('OPENAI_API_KEY')))
print('ANTHROPIC_API_KEY configured:', bool(os.environ.get('ANTHROPIC_API_KEY')))
ENABLE_NEWS_RESEARCH = bool(os.environ.get('TAVILY_API_KEY'))
print('Optional Tavily research enabled:', ENABLE_NEWS_RESEARCH)
if not os.environ.get('OPENAI_API_KEY'):
    raise RuntimeError('This notebook uses the OpenAI model configured in jmodels.py. Set OPENAI_API_KEY in m5/.env.')

## Local mock mail MCP service

The course's MCP feature stays intact. This cell starts its offline mail service on `127.0.0.1:5002`; it exposes tools only to the graph, not a user interface. The final cleanup cell stops the process that this notebook started.

In [ ]:
import atexit
import socket
import subprocess
import time

MCP_DIR = SALES_DIR / 'mcp'
MAIL_URL = 'http://127.0.0.1:5002/mcp'
_mail_process = None

def _port_open(host='127.0.0.1', port=5002):
    try:
        with socket.create_connection((host, port), timeout=0.2):
            return True
    except OSError:
        return False

if not _port_open():
    _mail_process = subprocess.Popen(
        [sys.executable, 'mock_mail_server.py'],
        cwd=MCP_DIR,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
    )
    for _ in range(50):
        if _port_open():
            break
        if _mail_process.poll() is not None:
            raise RuntimeError(_mail_process.stderr.read())
        time.sleep(0.1)
    else:
        raise TimeoutError('Mock mail MCP service did not start.')
    print('Started local mock mail MCP service.')
else:
    print('Using the mock mail MCP service already running on port 5002.')

def stop_mock_mail_service():
    if _mail_process and _mail_process.poll() is None:
        _mail_process.terminate()
        _mail_process.wait(timeout=5)
        print('Stopped notebook-owned mock mail MCP service.')

atexit.register(stop_mock_mail_service)

## Build the in-process deep agent

This preserves the course architecture: shared filesystem backend, `AGENTS.md`, task skills, code interpreter, and the four specialists. The installed course version of Deep Agents already adds `TodoListMiddleware`; adding it a second time is invalid. A checkpoint saver is required for the two human-approval gates.

In [ ]:
from deepagents import FilesystemPermission, MemoryMiddleware, create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_quickjs import CodeInterpreterMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

try:
    from jmodels import model  # local override, if supplied
except ModuleNotFoundError:
    from models import model  # course default
import subagents as course_subagents
from subagents import GENRE_PROMPT
from tools.chart import render_pie_chart
from tools.sql import query_chinook

SYSTEM_PROMPT = (
    'You are a sales assistant for Jane Peacock, a Sales Support Agent at Chinook. '
    'Follow the operating manual loaded from memory and the matching /skills playbook. '
    'You are running inside a Jupyter notebook: never create HTML and never direct the user to a browser. '
    'Write completed deliverables under /outputs/. Save newsletters as Markdown, not HTML.'
)

# Virtual paths keep /skills and AGENTS.md available to agent middleware.
backend = FilesystemBackend(root_dir=str(SALES_DIR), virtual_mode=True)
mail_client = MultiServerMCPClient({'mock-mail': {'transport': 'streamable-http', 'url': MAIL_URL}})
mail_tools = await mail_client.get_tools()

# Use the working OpenAI course model consistently. jmodels.py's optional Anthropic
# strong_model is intentionally not used: its configured credential was rejected.
course_subagents.model = model
course_subagents.strong_model = model

sales_assistant = create_deep_agent(
    model=model,
    # Mail tools stay solely on inbox-manager; the coordinator must delegate mail work.
    tools=[render_pie_chart],
    system_prompt=SYSTEM_PROMPT,
    subagents=course_subagents.build_subagents(backend, enable_search=ENABLE_NEWS_RESEARCH, mail_tools=mail_tools),
    skills=['/skills'],
    memory=['/AGENTS.md'],
    backend=backend,
    # Deep Agents 0.6.x supplies TodoListMiddleware by default.
    middleware=[CodeInterpreterMiddleware()],
    checkpointer=InMemorySaver(),
    name='chinook-sales-assistant-notebook',
)
print('Graph built. MCP tools:', [tool.name for tool in mail_tools])

## Notebook runner and approval gates

`run_task` calls the graph directly—there is no HTTP agent server. If a specialist requests `mail_create_draft` or `add_customer`, the result contains a LangGraph interrupt. Inspect it, then use `resume_approval` to approve, reject, or edit.

In [ ]:
from IPython.display import Image, Markdown, display
from pprint import pprint

def _message_text(message):
    content = getattr(message, 'content', message.get('content', '') if isinstance(message, dict) else '')
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return ''.join(part.get('text', '') for part in content if isinstance(part, dict))
    return str(content)

def show_latest_reply(result):
    for message in reversed(result.get('messages', [])):
        if getattr(message, 'type', None) == 'ai' or (isinstance(message, dict) and message.get('type') == 'ai'):
            display(Markdown(_message_text(message)))
            return

def show_interrupt(result):
    interrupts = result.get('__interrupt__', ())
    if interrupts:
        print('Approval required. Inspect the proposed action, then call resume_approval(...).')
        pprint(interrupts[0].value)
    return bool(interrupts)

async def run_task(prompt, thread_id):
    config = {'configurable': {'thread_id': thread_id}}
    result = await sales_assistant.ainvoke(
        {'messages': [{'role': 'user', 'content': prompt}]}, config=config
    )
    show_latest_reply(result)
    show_interrupt(result)
    return result, config

async def resume_approval(config, decision='approve', edited_action=None):
    if decision not in {'approve', 'reject', 'edit'}:
        raise ValueError('decision must be approve, reject, or edit')
    item = {'type': decision}
    if decision == 'edit':
        if not edited_action:
            raise ValueError('edit requires edited_action={name: ..., args: {...}}')
        item['edited_action'] = edited_action
    result = await sales_assistant.ainvoke(Command(resume={'decisions': [item]}), config=config)
    show_latest_reply(result)
    show_interrupt(result)
    return result

## 1. Territory report

The main agent delegates all database work to `chinook-analyst`, uses the interpreter for calculations, writes a Markdown report to `/outputs/`, and renders a PNG chart.

In [ ]:
territory_result, territory_config = await run_task(
    "How's my book of business looking? Build a territory report with a revenue-by-genre pie chart.",
    thread_id='territory-report',
)

for report in sorted(OUTPUTS_DIR.glob('territory_report-*.md')):
    print(report.name)
    display(Markdown(report.read_text()))
for chart in sorted(OUTPUTS_DIR.glob('territory_chart*.png')):
    display(Image(filename=str(chart)))

## 2. Optional parallel genre research and Markdown newsletter

If `TAVILY_API_KEY` is configured, one isolated `genre-researcher` runs for each genre through `asyncio.gather`, so the calls fan out concurrently. They use Tavily's API directly; no browser or HTML is involved. Each specialist saves raw notes under `/research/<genre>/sources.md`.

In [ ]:
import asyncio
import json
from datetime import date

TOP_GENRES_SQL = '''
SELECT g.Name AS genre, ROUND(SUM(il.UnitPrice * il.Quantity), 2) AS revenue
FROM InvoiceLine AS il
JOIN Track AS t ON t.TrackId = il.TrackId
JOIN Genre AS g ON g.GenreId = t.GenreId
GROUP BY g.GenreId, g.Name
ORDER BY revenue DESC
LIMIT 4
'''

from openai import RateLimitError

async def research_genre(genre, position):
    # A separate graph per genre makes the concurrency explicit and keeps raw notes isolated by path.
    from tools.search import internet_search
    researcher = create_deep_agent(
        model=model,
        tools=[internet_search],
        system_prompt=GENRE_PROMPT,
        backend=backend,
        permissions=[
            FilesystemPermission(operations=['read', 'write'], paths=['/research/**'], mode='allow'),
            FilesystemPermission(operations=['write'], paths=['/**'], mode='deny'),
        ],
        name=f'genre-researcher-{genre.lower().replace(" ", "-")}',
    )
    for attempt in range(5):
        try:
            state = await researcher.ainvoke({'messages': [{'role': 'user', 'content': f'Research {genre} for this week.'}]})
            return genre, _message_text(state['messages'][-1])
        except RateLimitError:
            if attempt == 4:
                raise
            # Keep asyncio.gather fan-out; jitter prevents every researcher from retrying together.
            delay = 10 * (attempt + 1) + 3 * position
            print(f'{genre}: rate limited; retrying in {delay}s')
            await asyncio.sleep(delay)

if not ENABLE_NEWS_RESEARCH:
    print('Skipped: set TAVILY_API_KEY and rerun this cell to enable the optional research feature.')
else:
    genres = [row['genre'] for row in json.loads(query_chinook.invoke({'sql': TOP_GENRES_SQL}))]
    researched = await asyncio.gather(*(research_genre(genre, position) for position, genre in enumerate(genres)))
    segments = '\n\n'.join(segment for _, segment in researched)
    newsletter = '# This Week in Music\n\nA concise update from Chinook.\n\n' + segments
    newsletter_path = OUTPUTS_DIR / f'newsletter-{date.today().isoformat()}.md'
    newsletter_path.write_text(newsletter, encoding='utf-8')
    print(f'Saved {newsletter_path.name}; researched concurrently: {genres}')
    display(Markdown(newsletter))

## 3. RFQ workflow with human approval

This resets the offline mailbox, processes the seeded RFQ, looks up prices through the database specialist, computes exact totals in the interpreter, gets a quote review, and pauses before saving a draft. Run the next cell, inspect the interrupt, then approve or reject it in the following cell. The same mechanism pauses before a new-customer insert.

In [ ]:
# Reset only the tiny mock mailbox. It does not modify the Chinook database.
subprocess.run([sys.executable, 'send_to_inbox.py', '--reset'], cwd=MCP_DIR, check=True)
rfq_result, rfq_config = await run_task(
    'Check the inbox for quote requests and process them. Follow the rfq-quote skill exactly.',
    thread_id='rfq-workflow',
)

In [ ]:
# Human-in-the-loop means the notebook must not auto-approve this action.
# After inspecting the previous cell, set APPROVAL to 'approve', 'reject', or 'edit' and rerun.
APPROVAL = None
if rfq_result.get('__interrupt__') and APPROVAL:
    rfq_result = await resume_approval(rfq_config, decision=APPROVAL)
elif rfq_result.get('__interrupt__'):
    print('No action taken. Set APPROVAL after reviewing the proposed customer or draft action.')

ledger = OUTPUTS_DIR / 'quotes_ledger.md'
if ledger.exists():
    display(Markdown(ledger.read_text()))

## Optional new-customer approval

To study the second gate without guessing customer data, ask the agent for an explicitly named new customer. It will pause before `add_customer`; use `resume_approval(customer_config, 'approve')` only after verifying the proposed fields. This changes the copied course database, so reject it unless you deliberately want the insert.

In [ ]:
# Example only: replace the details or leave this cell unrun.
# customer_result, customer_config = await run_task(
#     'Add new customer Ada Example (ada@example.test), company Example Records, in Seattle, USA. Confirm she does not already exist first.',
#     thread_id='new-customer',
# )
# await resume_approval(customer_config, decision='reject')

## Inspect notebook-native artifacts and clean up

All deliverables remain ordinary local files under `sales_assistant/outputs/`; no browser is needed to inspect them. The final call stops only the mock MCP service started by this notebook.

In [ ]:
for path in sorted(OUTPUTS_DIR.iterdir()):
    print(f'{path.name}: {path.stat().st_size:,} bytes')

# Run when finished. Safe to call more than once.
# stop_mock_mail_service()